In [3]:
from math_verify import parse, verify
from datasets import load_from_disk

ANSWER_FORCE_STRING = "\n\n**Final Answer**\n\\[\\boxed{"


def is_correct(example, trace_colname):
    """
    Evaluate if a generated trace produces the correct answer for a math problem.
    
    Uses math_verify to parse and compare solutions. Handles cases where the 
    answer forcing string splits the response.
    """
    trace = example[trace_colname]
    try:
        soln = parse(example["solution"])
        if ANSWER_FORCE_STRING in trace:
            # Handle answer forcing: try multiple ways to extract the answer
            parts = trace.split(ANSWER_FORCE_STRING)
            alt_ans1 = ANSWER_FORCE_STRING.join(parts[:-1])
            alt_ans2 = ANSWER_FORCE_STRING + parts[-1]
            res = verify(soln, parse(alt_ans2))
            # res = any(verify(soln, parse(ans)) for ans in [trace, alt_ans1, alt_ans2])
            # res = any(verify(soln, parse(ans)) for ans in [alt_ans1, alt_ans2])
        else:
            res = verify(soln, parse(trace))
    except:
        print(f"Error parsing trace: {trace} and comparing with solution: {example['solution']}")
        res = False
    return res


In [15]:
from pathlib import Path

path = Path('./experiments_gsm8k/traces')

# Lọc các entry là thư mục (is_dir)
subdirectories = sorted([d.name for d in path.iterdir() if d.is_dir()])

subdirectories

['eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'holdout',
 'tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'tau9.00e-01_lam3.16e-02_eps1.00e-02']

In [16]:
from pathlib import Path
from datasets import load_from_disk

gsm8k = './experiments_gsm8k/traces/'
gsm8k_paths = ['eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02',]

gsm8k_dict = {}
for path in gsm8k_paths:
    data = load_from_disk(gsm8k + path)
    n_corr = 0
    for example in data:
        res = is_correct(example, 'trace_af')
        n_corr += int(res)
    gsm8k_dict[path] = n_corr / len(data)

gsm8k_paired_results = {}

for key, score in gsm8k_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in gsm8k_paired_results:
        gsm8k_paired_results[config] = {}
        
    gsm8k_paired_results[config][role] = score

gsm8k_paired_results

{'tau1.00e+00_lam3.95e-02_eps1.00e-02': {'student': 0.445822994210091,
  'teacher': 0.6815550041356493},
 'tau9.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.5798180314309347,
  'teacher': 0.8891645988420182},
 'tau9.00e-01_lam2.37e-02_eps1.00e-02': {'student': 0.5599669148056244,
  'teacher': 0.8511166253101737},
 'tau9.00e-01_lam3.16e-02_eps1.00e-02': {'student': 0.4913151364764268,
  'teacher': 0.7998345740281224}}

In [3]:
from pathlib import Path
from datasets import load_from_disk

gsm8k = './experiments_gsm8k_default/traces/'
gsm8k_paths = ['eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02',]

gsm8k_dict = {}
for path in gsm8k_paths:
    data = load_from_disk(gsm8k + path)
    n_corr = 0
    for example in data:
        res = is_correct(example, 'trace_af')
        n_corr += int(res)
    gsm8k_dict[path] = n_corr / len(data)

gsm8k_paired_results = {}

for key, score in gsm8k_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in gsm8k_paired_results:
        gsm8k_paired_results[config] = {}
        
    gsm8k_paired_results[config][role] = score

gsm8k_paired_results

{'tau1.00e+00_lam3.95e-02_eps1.00e-02': {'student': 0.38626964433416044,
  'teacher': 0.6476426799007444},
 'tau9.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.6104218362282878,
  'teacher': 0.8966087675765095},
 'tau9.00e-01_lam2.37e-02_eps1.00e-02': {'student': 0.5574855252274608,
  'teacher': 0.8395368072787428},
 'tau9.00e-01_lam3.16e-02_eps1.00e-02': {'student': 0.5037220843672456,
  'teacher': 0.7890818858560794}}

In [7]:
from pathlib import Path
from datasets import load_from_disk

gsm8k = './experiments_gsm8k_topp95_topk50/traces/'
gsm8k_paths = ['eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02']

gsm8k_dict = {}
for path in gsm8k_paths:
    data = load_from_disk(gsm8k + path)
    n_corr = 0
    for example in data:
        res = is_correct(example, 'trace_af')
        n_corr += int(res)
    gsm8k_dict[path] = n_corr / len(data)

gsm8k_paired_results = {}

for key, score in gsm8k_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in gsm8k_paired_results:
        gsm8k_paired_results[config] = {}
        
    gsm8k_paired_results[config][role] = score

gsm8k_paired_results

{'tau1.00e+00_lam3.95e-02_eps1.00e-02': {'student': 0.4052936311000827,
  'teacher': 0.684863523573201},
 'tau9.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.5996691480562448,
  'teacher': 0.8775847808105872},
 'tau9.00e-01_lam2.37e-02_eps1.00e-02': {'student': 0.5641025641025641,
  'teacher': 0.8428453267162944},
 'tau9.00e-01_lam3.16e-02_eps1.00e-02': {'student': 0.5053763440860215,
  'teacher': 0.7866004962779156}}

In [8]:
gsm8k = './experiments_gsm8k_proxy1.5B/traces/'
gsm8k_paths = ['eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02']

gsm8k_dict = {}
for path in gsm8k_paths:
    data = load_from_disk(gsm8k + path)
    n_corr = 0
    for example in data:
        res = is_correct(example, 'trace_af')
        n_corr += int(res)
    gsm8k_dict[path] = n_corr / len(data)

gsm8k_paired_results = {}

for key, score in gsm8k_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in gsm8k_paired_results:
        gsm8k_paired_results[config] = {}
        
    gsm8k_paired_results[config][role] = score

gsm8k_paired_results

{'tau1.00e+00_lam3.95e-02_eps1.00e-02': {'student': 0.07692307692307693,
  'teacher': 0.6559139784946236},
 'tau9.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.6038047973531845,
  'teacher': 0.8999172870140613},
 'tau9.00e-01_lam2.37e-02_eps1.00e-02': {'student': 0.5161290322580645,
  'teacher': 0.8817204301075269},
 'tau9.00e-01_lam3.16e-02_eps1.00e-02': {'student': 0.4077750206782465,
  'teacher': 0.782464846980976}}

In [ ]:
gsm8k = './exps/experiments_gsm8k_proxy1.5B/traces/'
gsm8k_paths = [
 'eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02']

gsm8k_dict = {}
for path in gsm8k_paths:
    data = load_from_disk(gsm8k + path)
    n_corr = 0
    for example in data:
        res = is_correct(example, 'trace_af')
        n_corr += int(res)
    gsm8k_dict[path] = n_corr / len(data)

gsm8k_paired_results = {}

for key, score in gsm8k_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in gsm8k_paired_results:
        gsm8k_paired_results[config] = {}
        
    gsm8k_paired_results[config][role] = score

gsm8k_paired_results

{'tau1.00e+00_lam3.95e-02_eps1.00e-02': {'student': 0.0934656741108354,
  'teacher': 0.6559139784946236},
 'tau9.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.5988420181968569,
  'teacher': 0.8999172870140613},
 'tau9.00e-01_lam3.16e-02_eps1.00e-02': {'student': 0.3325062034739454,
  'teacher': 0.782464846980976},
 'tau9.00e-01_lam2.37e-02_eps1.00e-02': {'teacher': 0.8817204301075269}}

In [2]:
math = './experiments_math/traces/'
math_paths = [
 'eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 ]


math_dict = {}
for path in math_paths:
    data = load_from_disk(math + path)
    n_corr = 0
    for example in data:
        res = is_correct(example, 'trace_af')
        n_corr += int(res)
    math_dict[path] = n_corr / len(data)

math_paired_results = {}

for key, score in math_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in math_paired_results:
        math_paired_results[config] = {}
        
    math_paired_results[config][role] = score

math_paired_results

Timeout during comparison


{'tau1.00e+00_lam3.95e-02_eps1.00e-02': {'student': 0.1329861111111111,
  'teacher': 0.5989583333333334},
 'tau9.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.13333333333333333,
  'teacher': 0.5815972222222222},
 'tau9.00e-01_lam2.37e-02_eps1.00e-02': {'student': 0.13055555555555556,
  'teacher': 0.5791666666666667},
 'tau9.00e-01_lam3.16e-02_eps1.00e-02': {'student': 0.14027777777777778,
  'teacher': 0.5875}}

In [3]:
import re
from math_verify import parse, verify
from datasets import load_from_disk

mmlu = './experiments_mmlu/traces/'
mmlu_paths = ['eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02',]

ANSWER_FORCE_STRING = "\n\n**Final Answer**\n\\[\\boxed{"

mmlu_dict = {}
for path in mmlu_paths:
    data = load_from_disk(mmlu + path)
    n_corr = 0
    for example in data:
        trace = example['trace_af']
        target_text = example.get('solution', '') 
        boxed_match = re.search(r'\\boxed{(.+?)}', target_text)
        
        ground_truth_letter = ""
        if boxed_match:
            ground_truth_letter = boxed_match.group(1).strip().upper()
            
        options = {}
        for line in example['problem'].split('\n'):
            match = re.match(r'^([A-D])\.\s*(.*)', line.strip())
            if match:
                options[match.group(1)] = match.group(2).strip()

        true_value = options.get(ground_truth_letter, "").strip()

        soln = parse(example["solution"]) + parse('\\boxed{' + true_value + '}')
        
        if ANSWER_FORCE_STRING in trace:
            # Handle answer forcing: try multiple ways to extract the answer
            parts = trace.split(ANSWER_FORCE_STRING)
            alt_ans1 = ANSWER_FORCE_STRING.join(parts[:-1])
            alt_ans2 = ANSWER_FORCE_STRING + parts[-1]
            res = verify(soln, parse(alt_ans2))
            # res = any(verify(soln, parse(ans)) for ans in [trace, alt_ans1, alt_ans2])
            # res = any(verify(soln, parse(ans)) for ans in [alt_ans1, alt_ans2])
        else:
            res = verify(soln, parse(trace))

        n_corr += int(res)
    mmlu_dict[path] = n_corr / len(data)

mmlu_paired_results = {}

for key, score in mmlu_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in mmlu_paired_results:
        mmlu_paired_results[config] = {}
        
    mmlu_paired_results[config][role] = score

mmlu_paired_results

{'tau1.00e+00_lam3.95e-02_eps1.00e-02': {'student': 0.4250434782608696,
  'teacher': 0.6156521739130435},
 'tau9.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.4528695652173913,
  'teacher': 0.6462608695652174},
 'tau9.00e-01_lam2.37e-02_eps1.00e-02': {'student': 0.463304347826087,
  'teacher': 0.6493913043478261},
 'tau9.00e-01_lam3.16e-02_eps1.00e-02': {'student': 0.4048695652173913,
  'teacher': 0.632}}

In [3]:
import re
from math_verify import parse, verify
from datasets import load_from_disk

mmlu = './exps/experiments_mmlu_default/traces/'
mmlu_paths = ['eval_student_tau1.16e+00_lam0.00e+00_eps0.00e+00',
 'eval_teacher_tau1.16e+00_lam0.00e+00_eps0.00e+00',]

ANSWER_FORCE_STRING = "\n\n**Final Answer**\n\\[\\boxed{"

mmlu_dict = {}
for path in mmlu_paths:
    data = load_from_disk(mmlu + path)
    n_corr = 0
    for example in data:
        trace = example['trace_af']
        target_text = example.get('solution', '') 
        boxed_match = re.search(r'\\boxed{(.+?)}', target_text)
        
        ground_truth_letter = ""
        if boxed_match:
            ground_truth_letter = boxed_match.group(1).strip().upper()
            
        options = {}
        for line in example['problem'].split('\n'):
            match = re.match(r'^([A-D])\.\s*(.*)', line.strip())
            if match:
                options[match.group(1)] = match.group(2).strip()

        true_value = options.get(ground_truth_letter, "").strip()

        soln = parse(example["solution"]) + parse('\\boxed{' + true_value + '}')
        
        if ANSWER_FORCE_STRING in trace:
            # Handle answer forcing: try multiple ways to extract the answer
            parts = trace.split(ANSWER_FORCE_STRING)
            alt_ans1 = ANSWER_FORCE_STRING.join(parts[:-1])
            alt_ans2 = ANSWER_FORCE_STRING + parts[-1]
            res = verify(soln, parse(alt_ans2))
            # res = any(verify(soln, parse(ans)) for ans in [trace, alt_ans1, alt_ans2])
            # res = any(verify(soln, parse(ans)) for ans in [alt_ans1, alt_ans2])
        else:
            res = verify(soln, parse(trace))

        n_corr += int(res)
    mmlu_dict[path] = n_corr / len(data)

mmlu_paired_results = {}

for key, score in mmlu_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in mmlu_paired_results:
        mmlu_paired_results[config] = {}
        
    mmlu_paired_results[config][role] = score

mmlu_paired_results

{'tau1.16e+00_lam0.00e+00_eps0.00e+00': {'student': 0.4368695652173913,
  'teacher': 0.6431304347826087}}

In [6]:
import re
from math_verify import parse, verify
from datasets import load_from_disk

mmlu = './experiments_mmlu/traces/'
mmlu_paths = [
 'eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02',
'eval_student_tau5.00e-01_lam1.58e-02_eps1.00e-02',
 ]

ANSWER_FORCE_STRING = "\n\n**Final Answer**\n\\[\\boxed{"

mmlu_dict = {}
for path in mmlu_paths:
    data = load_from_disk(mmlu + path)
    n_corr = 0
    for example in data:
        trace = example['trace_af']
        target_text = example.get('solution', '') 
        boxed_match = re.search(r'\\boxed{(.+?)}', target_text)
        
        ground_truth_letter = ""
        if boxed_match:
            ground_truth_letter = boxed_match.group(1).strip().upper()
            
        options = {}
        for line in example['problem'].split('\n'):
            match = re.match(r'^([A-D])\.\s*(.*)', line.strip())
            if match:
                options[match.group(1)] = match.group(2).strip()

        true_value = options.get(ground_truth_letter, "").strip()

        soln = parse(example["solution"]) + parse('\\boxed{' + true_value + '}')
        
        if ANSWER_FORCE_STRING in trace:
            # Handle answer forcing: try multiple ways to extract the answer
            parts = trace.split(ANSWER_FORCE_STRING)
            alt_ans1 = ANSWER_FORCE_STRING.join(parts[:-1])
            alt_ans2 = ANSWER_FORCE_STRING + parts[-1]
            res = verify(soln, parse(alt_ans2))
            # res = any(verify(soln, parse(ans)) for ans in [trace, alt_ans1, alt_ans2])
            # res = any(verify(soln, parse(ans)) for ans in [alt_ans1, alt_ans2])
        else:
            res = verify(soln, parse(trace))

        # if not res:
        #     print(soln, alt_ans2)
        #     print(trace)

        n_corr += int(res)
    mmlu_dict[path] = n_corr / len(data)

mmlu_paired_results = {}

for key, score in mmlu_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in mmlu_paired_results:
        mmlu_paired_results[config] = {}
        
    mmlu_paired_results[config][role] = score

mmlu_paired_results

{'tau1.00e+00_lam3.95e-02_eps1.00e-02': {'student': 0.4791666666666667,
  'teacher': 0.6986111111111111},
 'tau9.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.5215277777777778,
  'teacher': 0.7059027777777778},
 'tau9.00e-01_lam2.37e-02_eps1.00e-02': {'student': 0.49236111111111114,
  'teacher': 0.7041666666666667},
 'tau9.00e-01_lam3.16e-02_eps1.00e-02': {'student': 0.521875,
  'teacher': 0.7041666666666667},
 'tau5.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.5142361111111111}}

In [1]:
import re
from math_verify import parse, verify
from datasets import load_from_disk

mmlu = './experiments_mmlu/traces/'
mmlu_paths = [
 'eval_student_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_student_tau9.00e-01_lam3.16e-02_eps1.00e-02',
 'eval_teacher_tau1.00e+00_lam3.95e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam1.58e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam2.37e-02_eps1.00e-02',
 'eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02',
'eval_student_tau5.00e-01_lam1.58e-02_eps1.00e-02',
 ]

ANSWER_FORCE_STRING = "\n\n**Final Answer**\n\\[\\boxed{"

mmlu_dict = {}
for path in mmlu_paths:
    data = load_from_disk(mmlu + path)
    n_corr = 0
    for example in data:
        trace = example['trace_af']
        soln = parse(example["solution"])
        
        if ANSWER_FORCE_STRING in trace:
            parts = trace.split(ANSWER_FORCE_STRING)
            alt_ans2 = ANSWER_FORCE_STRING + parts[-1]
            res = verify(soln, parse(alt_ans2))
        else:
            res = verify(soln, parse(trace))


        n_corr += int(res)
    mmlu_dict[path] = n_corr / len(data)

mmlu_paired_results = {}

for key, score in mmlu_dict.items():
    parts = key.split('_tau')
    role = parts[0].replace('eval_', '') # 'student' hoặc 'teacher'
    config = 'tau' + parts[1]
    
    if config not in mmlu_paired_results:
        mmlu_paired_results[config] = {}
        
    mmlu_paired_results[config][role] = score

mmlu_paired_results

{'tau1.00e+00_lam3.95e-02_eps1.00e-02': {'student': 0.47152777777777777,
  'teacher': 0.6951388888888889},
 'tau9.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.51875,
  'teacher': 0.7045138888888889},
 'tau9.00e-01_lam2.37e-02_eps1.00e-02': {'student': 0.48715277777777777,
  'teacher': 0.7010416666666667},
 'tau9.00e-01_lam3.16e-02_eps1.00e-02': {'student': 0.5152777777777777,
  'teacher': 0.7010416666666667},
 'tau5.00e-01_lam1.58e-02_eps1.00e-02': {'student': 0.5090277777777777}}

In [2]:
from datasets import load_from_disk

data = load_from_disk("/media/volume/ES_volumne/LLM_Distillation/baseline-distillm/exps/experiments_gsm8k_default/traces/eval_teacher_tau9.00e-01_lam3.16e-02_eps1.00e-02")

dataset = data.shuffle(seed=42)
sample = dataset.select(range(20))

for item in sample:
    print("---------"*50)
    print(item['trace_af'])
    print(item['solution'])
    print("---------"*50)
    print('\n')

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
<｜begin▁of▁sentence｜>You are a math teacher. You will be given a math problem and you will solve it step by step.
You will output your final solution like \boxed{ANSWER}. Be sure to include relevant units within the brackets and fully evaluate arithmetic expressions.
<｜User｜>Craig has 2 twenty dollar bills. He buys six squirt guns for $2 each.  He also buys 3 packs of water balloons for $3 each.  How much money does he have left?
<｜Assistant｜><think>
Craig starts with 2 twenty dollar bills. Each bill is worth $20, so we multiply $20 by 2 to fi

In [ ]:
from _distill.utils import load_dataset, load_gsm8k, load_hendrycks_math_dataset, load_mmlu, SYSTEM_PROMPT
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

# dataset = load_gsm8k(split='test')
# dataset = load_hendrycks_math_dataset(split='test')
dataset = load_mmlu(split='test')

def preprocess_function(examples):
    # Create chat messages with system prompt and user problem
    messages = [[{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": problem.strip() + "\n"}] for problem in examples["problem"]]
    # Apply chat template to get tokenized inputs
    tokens = [toks for toks in tokenizer.apply_chat_template(messages, add_generation_prompt=True)]
    seq_lengths = [len(toks) for toks in tokens]
    return {"input_ids": tokens, "seq_lengths": seq_lengths}

proc_dataset = dataset.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    desc="Preprocessing dataset",
    load_from_cache_file=True,
)

max_prompt_length = 128
max_samples = 2880




proc_dataset = proc_dataset.filter(lambda x: x["seq_lengths"] <= max_prompt_length)
proc_dataset = proc_dataset.take(min(max_samples, len(proc_dataset)))

print(len(proc_dataset))


Preprocessing dataset (num_proc=4):   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

2880
